# Plantilla: regresión lineal (agnóstica al dataset)

Notebook **base** para predecir una variable **numérica continua** a partir de un CSV local. No está ligado a un dataset concreto: solo cambias rutas, columnas y, si quieres, la lista en `build_models()` (sklearn + XGBoost + CatBoost).

## Pasos principales (ejecutar en orden)

| Paso | Sección | Qué haces |
|------|---------|-----------|
| **0** | Helpers | Imports y funciones (`infer_feature_columns`, `infer_column_types`, `build_preprocess`, …). |
| **1** | Explorar CSV | `PREVIEW_PATH`, `PREVIEW_SEP` — nombres, tipos, faltantes, sugerencia numéricas/categóricas. |
| **2** | CONFIG | `DATA_PATH`, `TARGET_COL`, `DROP_COLS`; opcional `FEATURE_COLS`, `NUMERIC_COLS`, `CATEGORICAL_COLS`; `build_models()`. |
| **3** | Carga | `pd.read_csv` con los parámetros de CONFIG. |
| **4** | Calidad de datos | Distribución del target y faltantes. |
| **5** | Visualización | Histograma del target y scatter con una feature. |
| **6** | Split | Separar **X** e **y**; `train_test_split` (test 20 % por defecto). |
| **7** | Preprocesado | `ColumnTransformer`: numéricas → imputer + escalar; categóricas → imputer + one-hot. |
| **8** | Comparar modelos | Mismo preprocesado para todos los algoritmos; tabla y gráfico (orden por **R²**). |
| **9** | Mejor modelo | Scatter real vs predicho + matriz de confusión (redondeo o bins). |

El preprocesado va **dentro** del `Pipeline` con el estimador: `fit` solo en train, `transform` en test (evita *data leakage*). Columnas de **X** que no estén en numéricas ni categóricas se descartan (`remainder="drop"`).

### Primera vez con tu CSV

1. Copia el archivo a `data/`.
2. Ejecuta el paso **1** hasta que el `head()` se vea bien (prueba `,`, `;` o `\t` en el separador).
3. Rellena el paso **2** con el mismo path y separador.
4. Ejecuta del paso **3** al **9** sin saltar celdas.

**Ejemplos ya resueltos:** `01-regresion-lineal-wine-quality-red.ipynb` · `01-regresion-lineal-auto-mpg.ipynb` · `01-regresion-lineal-diabetes.ipynb`

> Ejecuta Jupyter desde `07.b-ejemplos-supervisados/` para que `data/...` resuelva bien.



In [ ]:
# =============================================================================
# Helpers — funciones reutilizables (misma lógica en todo el benchmark)
# =============================================================================
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer  # pipeline distinto por tipo de columna
from sklearn.impute import SimpleImputer  # rellenar NaN antes de escalar/codificar
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline  # encadena: preprocesado → modelo
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Las tres funciones siguientes se usan en varios pasos del notebook:
#   1. infer_feature_columns  → arma la lista de columnas X (predictores)
#   2. infer_column_types     → separa X en numéricas y categóricas
#   3. build_preprocess       → crea el ColumnTransformer (imputer + scaler / one-hot)



## 1. Explorar el CSV (antes de CONFIG)

Pon aquí la ruta de **tu** archivo. No rellenes CONFIG todavía: primero mira nombres de columnas, tipos y nulos.

Si una sola columna contiene todo el CSV, prueba otro `PREVIEW_SEP` (`","`, `";"`, `"\t"`).


In [ ]:
# --- Paso 1: explorar SIN tocar CONFIG todavía ---
PREVIEW_PATH = "data/mi_dataset.csv"  # ruta a tu CSV
PREVIEW_SEP = ","  # separador: ","  |  ";"  |  "\t"

# Carga provisional solo para inspeccionar estructura
df_preview = pd.read_csv(PREVIEW_PATH, sep=PREVIEW_SEP)

print(f"Filas: {len(df_preview):,}  |  Columnas: {len(df_preview.columns)}")
print("\n--- Nombres de columnas (índice : nombre) ---")
for i, col in enumerate(df_preview.columns):
    print(f"  {i:2d}: {col!r}")

print("\n--- Tipos de datos (dtypes) ---")
print(df_preview.dtypes)

print("\n--- Primeras filas ---")
display(df_preview.head())

# Faltantes: el pipeline imputará después; aquí solo diagnosticamos
print("\n--- Valores faltantes por columna ---")
missing = df_preview.isna().sum()
if missing.any():
    display(missing[missing > 0].to_frame("nulos"))
else:
    print("No hay valores faltantes.")

# Ayuda para rellenar NUMERIC_COLS / CATEGORICAL_COLS en CONFIG
_num = df_preview.select_dtypes(include=[np.number]).columns.tolist()
_cat = df_preview.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("\n--- Sugerencia automática de tipos ---")
print("Numéricas (int/float):", _num)
print("Categóricas (object/category/bool/string):", _cat)
print(
    "\n>>> Siguiente: en CONFIG pon DATA_PATH, CSV_SEP iguales y elige TARGET_COL "
    "(columna numérica continua a predecir)."
)



## 2. CONFIG — adaptar a tu dataset

Copia los valores decididos en la exploración. **Solo esta sección** (y opcionalmente `build_models()`) cambia entre proyectos.


In [ ]:
# ========== Paso 2: CONFIG — único bloque que cambia entre datasets ==========
DATA_PATH = "data/mi_dataset.csv"  # mismo path que PREVIEW_PATH
CSV_SEP = ","  # mismo separador que PREVIEW_SEP

TARGET_COL = "nombre_columna_objetivo"  # variable numérica continua (precio, mpg, quality…)

# Columnas que no deben usarse como features (ids, texto libre, leakage)
DROP_COLS = []  # ej. ["id", "car_name"]

# None = automático; o listas explícitas si la inferencia falla
FEATURE_COLS = None
NUMERIC_COLS = None
CATEGORICAL_COLS = None

TEST_SIZE = 0.2  # 20 % test, 80 % train
RANDOM_STATE = 42  # reproducibilidad del split y modelos
METRIC_PRINCIPAL = "r2"  # columna para ordenar la tabla (mayor = mejor en regresión)


def build_models():
    """Diccionario nombre → estimador. Comenta líneas para excluir modelos del benchmark."""
    from sklearn.ensemble import (
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.linear_model import Lasso, LinearRegression, Ridge
    from xgboost import XGBRegressor
    from catboost import CatBoostRegressor

    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
        "RandomForest": RandomForestRegressor(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "XGBoost": XGBRegressor(
            random_state=RANDOM_STATE, verbosity=0, n_estimators=100, n_jobs=-1
        ),
        "CatBoost": CatBoostRegressor(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }
    return models


MODELS = build_models()


## 3. Carga de datos

Vuelve a leer el CSV con los parámetros ya fijados en CONFIG.


In [ ]:
# --- Paso 3: carga definitiva con los parámetros de CONFIG ---
df = pd.read_csv(DATA_PATH, sep=CSV_SEP)
print("Shape:", df.shape)
df.head()



## 4. Calidad de datos

Diagnóstico de faltantes (el relleno real lo hace el pipeline en train).


In [ ]:
# --- Paso 4: calidad de datos (diagnóstico; el imputer actúa en el Pipeline) ---
print(df.info())
print("\nFaltantes por columna:")
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else "Sin valores faltantes")



## 5. Visualización rápida

Comprueba que el target sea numérico y tenga sentido para regresión.


In [ ]:
# --- Paso 5: visualización rápida del target (regresión) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma del target
df[TARGET_COL].hist(ax=axes[0], bins=20, edgecolor="black")
axes[0].set_title(f"Distribución de {TARGET_COL}")

# Relación lineal aproximada: primera feature numérica vs target
num_feat = df.select_dtypes(include=[np.number]).columns.drop(TARGET_COL, errors="ignore")
if len(num_feat):
    feat = num_feat[0]
    axes[1].scatter(df[feat], df[TARGET_COL], alpha=0.4)
    axes[1].set_xlabel(feat)
    axes[1].set_ylabel(TARGET_COL)

plt.tight_layout()
plt.show()



## 6. Separar X / y y split train-test

En regresión no hace falta `stratify` (solo en clasificación).


In [ ]:
# --- Paso 6: separar features (X), target (y) y dividir train / test ---
feature_cols = infer_feature_columns(df, TARGET_COL, DROP_COLS, FEATURE_COLS)
X = df[feature_cols]
y = df[TARGET_COL]

# Clasificación de columnas para el ColumnTransformer
numeric_cols, categorical_cols = infer_column_types(X, NUMERIC_COLS, CATEGORICAL_COLS)
print("Numéricas:", numeric_cols)
print("Categóricas:", categorical_cols)

# En regresión no usamos stratify (solo tiene sentido con clases discretas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)



## 7. Preprocesado (compartido por todos los modelos)

Un solo `ColumnTransformer` para todos los modelos del benchmark.


In [ ]:
# --- Paso 7: definir el preprocesador (mismo objeto para todos los modelos) ---
preprocess = build_preprocess(numeric_cols, categorical_cols)
preprocess  # muestra la estructura: ramas num y cat



## 8. Comparar modelos

Entrena cada entrada de `MODELS` con el mismo split y el mismo preprocesado. Puede tardar varios minutos.


In [ ]:
# --- Paso 8: métricas y benchmark de modelos ---

def regression_metrics(y_true, y_pred):
    """MAE, RMSE y R² en el conjunto indicado (aquí: test)."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}


def evaluate_models(models, preprocess, X_train, X_test, y_train, y_test):
    """Entrena un Pipeline por modelo y devuelve tabla comparativa."""
    rows = []
    for name, estimator in models.items():
        # Pipeline = preprocesado + modelo (fit solo con datos de train)
        pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        rows.append({"modelo": name, **regression_metrics(y_test, y_pred)})
    return pd.DataFrame(rows).sort_values(METRIC_PRINCIPAL, ascending=False)


results = evaluate_models(MODELS, preprocess, X_train, X_test, y_train, y_test)
display(results.round(4))

ax = results.plot(x="modelo", y=METRIC_PRINCIPAL, kind="barh", legend=False, figsize=(8, 5))
ax.set_xlabel("R² (test)")
ax.set_title("Comparación de modelos — regresión")
plt.tight_layout()
plt.show()



## 9. Detalle del mejor modelo

Gráfico **real vs predicho** en el conjunto de test.


In [ ]:
# --- Paso 9: detalle del mejor modelo según METRIC_PRINCIPAL ---
best_name = results.iloc[0]["modelo"]
print(f"Mejor modelo (test): {best_name}")

best_est = MODELS[best_name]
best_pipe = Pipeline([("preprocess", preprocess), ("model", best_est)])
best_pipe.fit(X_train, y_train)
y_pred_best = best_pipe.predict(X_test)

# Scatter: cada punto es una fila de test (eje x = real, eje y = predicho)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_best, alpha=0.5, edgecolors="k", linewidths=0.3)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
ax.plot(lims, lims, "r--", lw=1)  # línea ideal y=x
ax.set_xlabel("Valor real")
ax.set_ylabel("Predicción")
ax.set_title(f"{best_name}: real vs predicho (test)")
plt.tight_layout()
plt.show()

# Matriz de confusión: útil si el target es entero/discreto; si es muy continuo, usa bins
from sklearn.metrics import ConfusionMatrixDisplay

y_true_arr = np.asarray(y_test)
y_pred_arr = np.asarray(y_pred_best)

if pd.Series(y_train).nunique() <= 25:
    y_true_cm = np.round(y_true_arr).astype(int)
    y_pred_cm = np.round(y_pred_arr).astype(int)
    cm_note = "valores redondeados"
else:
    n_bins = 5
    bin_edges = np.unique(np.quantile(y_train, np.linspace(0, 1, n_bins + 1)))
    if len(bin_edges) < 2:
        bin_edges = np.linspace(float(np.min(y_train)), float(np.max(y_train)), n_bins + 1)
    y_true_cm = np.digitize(y_true_arr, bin_edges[1:-1])
    y_pred_cm = np.digitize(y_pred_arr, bin_edges[1:-1])
    cm_note = f"{len(bin_edges) - 1} intervalos (cuantiles de train)"

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_true_cm, y_pred_cm, ax=ax)
ax.set_title(f"Matriz de confusión ({cm_note}) — {best_name}")
plt.tight_layout()
plt.show()



## Checklist: nuevo dataset

1. CSV en `data/` → explorar (sección 1) → CONFIG (sección 2).
2. `TARGET_COL` debe ser **numérica continua** (precio, cantidad, puntuación…).
3. `DROP_COLS`: ids, texto libre, columnas que no deben influir en la predicción.
4. Si la detección automática de tipos falla, define `NUMERIC_COLS` / `CATEGORICAL_COLS`.
5. Opcional: comenta modelos en `build_models()` para acortar tiempo.
6. Ejecuta todas las celdas y compara la tabla (R², MAE, RMSE).
